A. Understand the dataset


In [1]:
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

patient = pd.read_csv(r"C:\2)Projects\ICU_Patient_Predict_Risk_Analysis\dataset\patient.csv")


patient_clean = patient.copy()

# 1 .shape - rows and columns
patient.shape
patient_clean.shape


(2520, 29)

In [ ]:
patient.info()

In [ ]:
# 2 columns
patient.columns.tolist()

In [ ]:
# 3. Data types
patient.dtypes

In [ ]:
# 4. First/last few rows
patient.head()

In [ ]:
patient.tail()

In [ ]:
# 5. Basic statistical summary
patient.describe(include="all")


In [ ]:
'''
now need to classify the above columns into  ID columns

#  ID columns id_cols

# categoricla columns
# continuous  or numerical categories
# time or date columns
# Target candidate columns


'''

In [ ]:

#  id columns



id_cols = [
    "patientunitstayid",
    "patienthealthsystemstayid",
    "hospitalid",
    "wardid",
    "uniquepid"
]

# categoricla columns

categorical_cols = [
    "gender",
    "ethnicity",
    "apacheadmissiondx",
    "hospitaladmitsource",
    "hospitaldischargelocation",
    "hospitaldischargestatus",
    "unittype",
    "unitadmitsource",
    "unitstaytype",
    "unitdischargelocation",
    "unitdischargestatus"
]



# continuous  or numerical categories

continuous_cols = [
    "age",
    "admissionheight",
    "hospitaladmitoffset",
    "hospitaldischargeoffset",
    "admissionweight",
    "dischargeweight",
    "unitvisitnumber",
    "unitdischargeoffset"
]


# time or date columns
time_cols = [
    "hospitaladmittime24",
    "hospitaldischargeyear",
    "hospitaldischargetime24",
    "unitadmittime24",
    "unitdischargetime24"
]

# Target candidate columns

target_candidate_cols = [
    "hospitaldischargestatus",
    "unitdischargestatus"
]



for col in target_candidate_cols:
    print(f"\n===== {col} =====")
    print(patient[col].value_counts(dropna=False))

patient["mortality"] = patient["hospitaldischargestatus"].map({
    "Alive": 0,
    "Expired": 1
})

patient["mortality"].value_counts(dropna=False)
patient["mortality"].value_counts(normalize=True, dropna=True) * 100


'''
   NOTE
Those 28 missing target values must not be imputed.

For supervised ML, we don't know their true outcome, so later we'll exclude those rows from model training/evaluation:

Unknown target → remove from modeling dataset
'''

In [ ]:
#  Numerical columns
num_cols = patient.select_dtypes(include="number").columns.tolist()
#or use this line  num_cols = patient.select_dtypes(include=["int64","float"]).columns.tolist()
print(num_cols)

# Categorical columns
cat_cols = patient.select_dtypes(include=["object", "category"]).columns.tolist()
print(cat_cols)

#  Datetime columns
date_cols = patient.select_dtypes(include="datetime").columns.tolist()
print(date_cols)

# 7. ID columns
id_cols = [col for col in patient.columns if "id" in col.lower()]
print(id_cols)



B. Data quality checks


In [ ]:
# 8. Missing values
patient.isnull().sum()


In [ ]:
#  Duplicate rows
patient.duplicated().sum()

In [ ]:
# 13. Outliers — IQR method  the below code is time taking process so 


# 1. Calculate Q1, Q3, and IQR for all columns simultaneously
q1 = patient[num_cols].quantile(0.25)
q3 = patient[num_cols].quantile(0.75)
iqr = q3 - q1

# 2. Check all values against the fences across the entire DataFrame matrix
is_outlier = (patient[num_cols] < (q1 - 1.5 * iqr)) | (patient[num_cols] > (q3 + 1.5 * iqr))

# 3. Sum along the columns to get the final dictionary
outliers = is_outlier.sum().to_dict()

print(outliers)




In [ ]:

# 15. Check whether IDs are unique
id_uniqueness = patient[id_cols].nunique()
print(id_uniqueness)

In [ ]:
# Id type COLUMNS:
# Check uniqueness
patient[id_cols].nunique()



In [ ]:
# Check duplicate IDs
patient[id_cols].duplicated().sum()

In [ ]:
# Check uniqueness percentage
patient[id_cols].nunique() / len(patient) * 100

In [ ]:
# Check ID values
patient[id_cols].head()

In [ ]:
#2 ) CATEGORICAL COLUMNS CHECK

In [ ]:


# Number of categories

print("  \n ----------  Number of categories ---------")
print(patient[categorical_cols].nunique())

# Category frequency
print( "\n ----------  Category frequency ---------")
for col in categorical_cols:
    print("\n col \n")
    print(patient[col].value_counts(dropna=False))

# Category percentage
print("\n ----------  Category percentage  ---------")
for col in categorical_cols:
    print("\n col")
    print("\n")
    print(patient[col].value_counts(normalize=True, dropna=False) * 100)

# Missing values
print("\n ----------  Missing values    --------")
print(patient[categorical_cols].isnull().sum())





In [ ]:
#  3 NUMERICAL OR CONTINUOUS COLUMNS

# treatment because of continuos varibles required

patient_clean["age"] = patient_clean["age"].replace("> 89", 89)

patient_clean["age"] = pd.to_numeric(
    patient_clean["age"],
    errors="coerce"
)

# Basic statistics
print("Basic statistics \n")
print(patient_clean[continuous_cols].describe())


# Distribution
print("Distribution \n")
print (patient_clean[continuous_cols].hist(figsize=(12, 8)))

# Skewness
print("Skewness \n")
print(patient_clean[continuous_cols].skew())

print("--"*100)

# Outliers using IQR
print("Outliers using IQR \n")
for col in continuous_cols:
    Q1 = patient_clean[col].quantile(0.25)
    Q3 = patient_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((patient_clean[col] < Q1 - 1.5 * IQR) | 
                (patient_clean[col] > Q3 + 1.5 * IQR)).sum()
    print(col, outliers)

# Correlation
print(" Correlation \n")
print(patient_clean[continuous_cols].corr())

# Missing values
print("Missing values \n")
print(patient_clean[continuous_cols].isnull().sum())

In [ ]:
print(continuous_cols)
print(patient_clean[continuous_cols].dtypes)

In [ ]:
# Mean, median, std, min, max

print("Mean, median, std, min, max \n")
print(patient_clean[continuous_cols].agg(["mean", "median", "std", "min", "max"]))


4. Date / Time columns

In [ ]:
'''
#time or date columns
time_cols = [
    "hospitaladmittime24",
    "hospitaldischargeyear",
    "hospitaldischargetime24",
    "unitadmittime24",
    "unitdischargetime24"
]

'''

In [ ]:
#check data types

print(patient[time_cols].dtypes)

#check sample value 

for col in time_cols:
    print("\n", col)
    print(patient[col].value_counts(dropna=False).head(10))

In [ ]:
# now conver the above list into only time realted columns

# currently dont consider year

# here converting 4 time columns
clock_cols = [
    "hospitaladmittime24",
    "hospitaldischargetime24",
    "unitadmittime24",
    "unitdischargetime24"
]

#  
'''
For example 
hospitaladmittime24       18:26:00
hospitaldischargetime24   20:00:00
unitadmittime24            19:52:00
unitdischargetime24        20:00:00
hospitaldischargeyear      2015 

Because currently Pandas may see 18:26:00 as text.

Converting it tells Pandas:

"This is a time value, not ordinary text."

'''
#  So here Dong convrt data and Time.
# This allows you to perform time operations and extract hour/minute.

for col in clock_cols:
    patient[col] = pd.to_datetime(patient[col], format="%H:%M:%S", errors="coerce")
print("-"*50)
print(patient[col])
patient[clock_cols].dtypes

#Extract useful features

Since these contain only time, extract hour and minute. 
Year/month/day are not available in these four columns.


why extract again hour and minute?
This creates new columns:  when patient adimitted time and minute

complete flow:

"18:26:00"  (text)
      ↓
pd.to_datetime()
      ↓
18:26:00  (datetime understood by Pandas)
      ↓
.dt.hour / .dt.minute
      ↓
18 / 26   (new numerical features)

In [ ]:
for col in clock_cols:
    patient[col + "_hour"] = patient[col].dt.hour
    patient[col + "_minute"] = patient[col].dt.minute

patient[[col + "_hour" for col in clock_cols]].head()

In [ ]:
# check missing values  after conversion
patient[clock_cols].isnull().sum()

In [ ]:
# Check time distributions

#    This helps you understand whether admissions/discharges happen mostly during particular hours.

for col in clock_cols:
    print("\n", col)
    print(patient[col + "_hour"].value_counts().sort_index())

#Chronological order and time gaps

hospitaldischargetime24 - hospitaladmittime24 

Note: we can not do this because  only time is available


Example:

Admission  = 23:00
Discharge  = 02:00

The discharge may have happened the next day.

For chronological sequence and duration, use your offset columns instead:

In [ ]:
# so created offset columns

# offset columns are used  These columns represent time relative to an event, measured in minutes.

#For example, conceptually:

#hospitaladmitoffset       → 0 minutes
#hospitaldischargeoffset   → 300 minutes

#That means the patient stayed in the hospital for about 300 minutes.

#So offsets are useful for calculating duration and sequence.


offset_cols = [
    "hospitaladmitoffset",
    "hospitaldischargeoffset",
    "unitdischargeoffset"
]

patient[offset_cols].describe()

In [ ]:
patient["hospital_length"] = (
    patient["hospitaldischargeoffset"] -
    patient["hospitaladmitoffset"]
)
print(patient["hospital_length"])

In [ ]:
# TAGET VAUE ANALYSIS
target = "hospitaldischargestatus"

patient["mortality"] = patient["hospitaldischargestatus"].map({
    "Alive": 0,
    "Expired": 1
})

patient["mortality"].value_counts(dropna=False)
patient["mortality"].value_counts(normalize=True, dropna=True) * 100

print(patient["hospitaldischargestatus"])

## EDA

In [ ]:

'''

EDA
│

Univariate Analysis

├── Numerical distributions
│       └── Histogram
│
├── Categorical distributions
│       └── Countplot
│
├── Outliers
│       └── Boxplot
│
└── Time patterns
        └── Hour / year / temporal patterns


BI variate Analysis:

├── Numerical vs Target
│       └── Boxplot / distribution comparison
│
├── Categorical vs Target
│       └── Countplot / proportions
│

MUltivariate Analysis

├── Numerical vs Numerical
│       └── Correlation

'''

1. Numerical → Numerical

Understand relationships between numerical features.

Note  : use  patient_clean dataframe not patient dataframe

Because your age currently contains > 89, it may still be object.

For EDA only, create a temporary numeric version:  patient_clean dataframe

These are already covered in columnwise: these called Univariate EDA

2)Numerical summary statistics with describe() -> already covered
3) Numerical distributions
4) Check skewness
5)Numerical missing values
6) Numerical outliers

-> CATEGORICAL EDA
valucounts
All categorical columns together  find valu_counts
and  Percentage

In [ ]:
# treatment

patient_clean["age"] = patient_clean["age"].replace("> 89", 89)

patient_clean["age"] = pd.to_numeric(
    patient_clean["age"],
    errors="coerce"
)

#Univariate Aanlysis:
Plots

 OUTLIER VISUALIZATION  ( For Continuous columns)

In [ ]:
for col in continuous_cols:
    sns.boxplot(x=patient_clean[col])
    plt.title(f"Outliers - {col}")
    plt.show()

  Numerical Distributions

In [ ]:
for col in continuous_cols:
    sns.histplot(data=patient_clean, x=col, kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

2) Categorical

  Categorical Distrbution

In [ ]:
for col in categorical_cols:
    patient_clean[col].value_counts().plot(kind="bar")
    plt.title(f"Distribution of {col}")
    plt.show()

3) Time-related EDA

For your eICU patient.csv, you can analyze admission/discharge time patterns.


sns.countplot(data=patient_clean, x="hospitaladmittime24")
plt.show()

  II ) BI variate Analysis

1. Numerical(continuous) → Target     (groupby  and  boxplot )

If your target is categorical, compare numerical features across target classes.



In [ ]:
# first do groupby  later do box plot

for col in continuous_cols:

    print("\n==============================")
    print(col)
    print("==============================")

    print(
        patient_clean.groupby(target)[col].describe()
    )

    sns.boxplot(x=patient_clean[target], y=patient_clean[col])
    plt.title(f"{col} vs {target}")
    plt.show()

2. categorical    crosstab count plot

In [ ]:
for col in categorical_cols:

    print("\n==============================")
    print(col, "vs", target)
    print("==============================")

    print(
        pd.crosstab(
            patient[col],
            patient[target],
            normalize="index"
        ).mul(100)
    )

    # Categorical and Target

for col in categorical_cols:
    sns.countplot(data=patient, x=col, hue=target)
    plt.title(f"{col} vs {target}")
    plt.xticks(rotation=45)
    plt.show()
  

MULTIVARAINT ANALYSIS

In [ ]:
#EDA :

#Missing values:
missing = patient.isnull().sum()

missing[missing > 0].sort_values(ascending=False)


In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": patient.isnull().sum(),
    "Missing_Percentage": patient.isnull().mean() * 100
})

missing_summary = (
    missing_summary[
        missing_summary["Missing_Count"] > 0
    ]
    .sort_values("Missing_Percentage", ascending=False)
)

missing_summary

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(
    patient_clean[continuous_cols].corr(),
    cmap="coolwarm",
    annot=True,
    fmt=".1f",
    center=0
)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
#Correlation:

corr = patient_clean[continuous_cols].corr()

for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):

        value = corr.iloc[i, j]

        if abs(value) < 0.3:
            category = "Weak"

        elif abs(value) < 0.7:
            category = "Moderate"

        else:
            category = "Strong"

        print(
            corr.columns[i],
            "<->",
            corr.columns[j],
            "=",
            round(value, 3),
            "->",
            category
        )

EDA  Findings:

1)Skewed variables

patient_eda[numeric_eda_cols].skew()

record something like:

admissionweight shows positive skewness.
age is relatively concentrated toward older patients.


2)6.3 Outliers

Based on boxplots:

Potential outliers observed in:
- admissionweight
- admissionheight

Again, only record what you actually see.

Don't delete them yet.



3)Missing Values

patient.isnull().sum().sort_values(ascending=False)


(
    patient.isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)






4) Target imbalance

This is particularly important for your mortality model.

patient[target].value_counts(dropna=False)

Percentage:

patient[target].value_counts(
    normalize=True,
    dropna=False
).mul(100)

Plot:

sns.countplot(
    data=patient,
    x=target
)

plt.title("Hospital Discharge Status Distribution")
plt.show()

You already found approximately:

Alive   → 2280
Expired → 212
NaN     → 28

from your earlier analysis.

So your target is clearly imbalanced, and the missing target rows will need to be handled during
the later data-cleaning/ML stage.





5) Correlations

Use:

corr

and identify pairs with relatively high correlation.

For example:

Potentially correlated:
admissionheight ↔ admissionweight



6) Potential leakage

This is very important for your project.

Your target is:

hospitaldischargestatus

Do not use future/outcome information as predictive features.

Review columns such as:

hospitaldischargetime24
hospitaldischargeoffset
hospitaldischargelocation
hospitaldischargestatus
unitdischargetime24
unitdischargeoffset
unitdischargelocation
unitdischargestatus

The target itself is excluded from features, and discharge-related information occurring after the prediction 
point should generally be excluded.




In [ ]:
print("Patient rows:", len(patient))
print("Patient unique ICU stays:",
      patient["patientunitstayid"].nunique())